# Credit Underwriting with Briefcase AI - Interactive Walkthrough

## Overview
This notebook demonstrates how to use Briefcase AI to create compliant audit trails for ML-based credit underwriting decisions under **ECOA/Reg B** regulations.

### What You'll Learn:
- How to capture AI credit decisions with immutable audit trails
- ECOA/Reg B compliance requirements for adverse action tracking
- How to simulate regulatory examiner queries
- Best practices for deterministic decision replay

### Regulatory Context:
- **Regulation**: ECOA/Reg B (Equal Credit Opportunity Act)
- **Regulator**: OCC/CFPB
- **Requirements**: Adverse action notices, non-discrimination, decision auditability

## Step 1: Setup and Imports

First, let's import the required libraries and set up our environment.

In [ ]:
import sys
import os
import uuid
import random
from datetime import datetime
from typing import Dict, Any

# Add shared module to path
_p = os.path.abspath('')
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, 'shared')):
    _p = os.path.dirname(_p)
if os.path.isdir(os.path.join(_p, 'shared')):
    sys.path.insert(0, os.path.join(_p, 'shared'))

try:
    import backend
    # Import SDK classes from backend (handles real SDK implementation)
    from backend import briefcase, DecisionSnapshot, Input, Output, SqliteBackend
    print("✓ Successfully imported Briefcase AI SDK")
except ImportError as e:
    print(f"✗ Error importing required modules: {e}")
    print("Please ensure the Briefcase AI SDK is installed and the shared backend is available")

## Step 2: Initialize Briefcase AI

Initialize the Briefcase AI SDK with the proper configuration.

In [ ]:
# Initialize Briefcase AI SDK
try:
    briefcase.init_with_config(2)  # 2 worker threads
    print("✓ Briefcase AI SDK initialized successfully")
except Exception as e:
    print(f"✗ Failed to initialize SDK: {e}")
    sys.exit(1)

# Get configured backend for audit trail storage
db_backend = backend.get_backend()
print("✓ SQLite backend configured for audit trail storage")

## Step 3: Simulate Credit Application Data

Let's create a realistic loan application with the data points typically used in credit underwriting.

In [ ]:
# Generate a loan application
applicant_id = str(uuid.uuid4())
application_data = {
    "applicant_id": applicant_id,
    "annual_income": 65000.0,
    "bureau_score": 685,  # Credit score
    "debt_to_income_ratio": 0.42,
    "loan_amount_requested": 25000.0,
    "loan_purpose": "auto",
    "behavioral_attributes_version": "v2.1.4"
}

print("**Details:** Loan Application Details:")
for key, value in application_data.items():
    print(f"  {key}: {value}")

print(f"\n**Insight:** Key Insights:")
print(f"  • DTI Ratio: {application_data['debt_to_income_ratio']} (target: <0.45)")
print(f"  • Credit Score: {application_data['bureau_score']} (good: >680)")
print(f"  • Loan-to-Income: {application_data['loan_amount_requested']/application_data['annual_income']:.1%}")

## Step 4: AI Credit Underwriting Model

This simulates an AI model that makes credit decisions. In production, this would be your actual ML model.

In [ ]:
def simulate_credit_underwriting_model(applicant_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Simulates an AI credit underwriting model decision.
    In production, this would be replaced with actual ML model inference.
    """
    # Extract key features
    annual_income = applicant_data["annual_income"]
    bureau_score = applicant_data["bureau_score"]
    dti_ratio = applicant_data["debt_to_income_ratio"]
    loan_amount = applicant_data["loan_amount_requested"]

    # Simple rule-based simulation for demonstration
    risk_score = 0.0
    
    # Credit score impact
    if bureau_score >= 720:
        risk_score += 0.4
    elif bureau_score >= 650:
        risk_score += 0.2
    else:
        risk_score -= 0.2

    # DTI ratio impact
    if dti_ratio <= 0.35:
        risk_score += 0.3
    elif dti_ratio <= 0.45:
        risk_score += 0.1
    else:
        risk_score -= 0.1

    # Income impact
    if annual_income >= 75000:
        risk_score += 0.2

    # Add some randomness to simulate model uncertainty
    risk_score += random.uniform(-0.1, 0.1)
    confidence = max(0.6, min(0.95, risk_score + 0.3))

    # Decision logic with ECOA compliance
    if risk_score >= 0.5:
        decision = "approve"
        approved_amount = loan_amount
        adverse_action_codes = None  # No adverse action needed
    elif risk_score >= 0.2:
        decision = "counter_offer"
        approved_amount = loan_amount * 0.75  # Reduce loan amount
        adverse_action_codes = ["credit_history", "debt_to_income_ratio"]
    else:
        decision = "decline"
        approved_amount = None
        adverse_action_codes = ["credit_history", "insufficient_income", "debt_to_income_ratio"]

    return {
        "decision": decision,
        "approved_amount": approved_amount,
        "adverse_action_reason_codes": adverse_action_codes,
        "confidence_score": round(confidence, 3),
        "model_version": "underwriting-model-v3.2.1",
        "decision_trace_id": str(uuid.uuid4())
    }

# Run the AI model
print("[AUTOMATED] Running AI underwriting model...")
model_output = simulate_credit_underwriting_model(application_data)

print(f"\n**Results:** Model Results:")
print(f"  • Decision: {model_output['decision'].upper()}")
print(f"  • Confidence: {model_output['confidence_score']}")
print(f"  • Model Version: {model_output['model_version']}")

if model_output['approved_amount']:
    print(f"  • Approved Amount: ${model_output['approved_amount']:,.2f}")
    
if model_output['adverse_action_reason_codes']:
    print(f"  • [WARNING] Adverse Action Codes: {model_output['adverse_action_reason_codes']}")
    print(f"    (Required for ECOA compliance)")

## Step 5: Create Immutable Audit Trail

This is the core of Briefcase AI - capturing the decision with all regulatory metadata in an immutable audit trail.

In [ ]:
# Prepare regulatory metadata for ECOA/Reg B compliance
regulatory_metadata = {
    "regulation": "ECOA/Reg B",
    "adverse_action_required": model_output["decision"] in ["decline", "counter_offer"],
    "bank_owns_model": True,
    "decision_timestamp": datetime.utcnow().isoformat(),
    "examiner_ready": True
}

print("**Details:** Regulatory Metadata:")
for key, value in regulatory_metadata.items():
    print(f"  • {key}: {value}")

# Create DecisionSnapshot using Briefcase AI
try:
    decision_snapshot = backend.create_decision_snapshot(
        function_name="credit_underwriting_decision",
        inputs=application_data,
        outputs=model_output,
        metadata=regulatory_metadata,
        input_types={
            "annual_income": "float",
            "bureau_score": "int",
            "debt_to_income_ratio": "float",
            "loan_amount_requested": "float"
        },
        output_types={
            "confidence_score": "float",
            "approved_amount": "float"
        }
    )
    print("\n[SUCCESS] Decision snapshot created successfully")
    
except Exception as e:
    print(f"\n[FAILED] Error creating decision snapshot: {e}")

## Step 6: Store in Immutable Audit Trail

Store the decision in the backend where it becomes part of an immutable audit trail.

In [ ]:
# Store decision in backend (creates immutable audit trail)
try:
    stored_decision_id = db_backend.save_decision(decision_snapshot)
    print(f"[SUCCESS] Decision stored in audit trail")
    print(f"[SECURED] Decision ID: {stored_decision_id}")
    print(f"\n**Insight:** This decision is now immutable and can be retrieved for regulatory purposes")
    
except Exception as e:
    print(f"[FAILED] Error storing decision: {e}")

## Step 7: Demonstrate Audit Trail Retrieval

Show how decisions can be retrieved from the audit trail for compliance purposes.

In [ ]:
print("**Analysis:** AUDIT TRAIL RETRIEVAL DEMONSTRATION")
print("=" * 50)

# Load decision back from backend
try:
    retrieved_decision = db_backend.load_decision(stored_decision_id)
    if retrieved_decision:
        print("[SUCCESS] Decision successfully retrieved from audit trail")
        
        # Display formatted audit summary
        backend.print_audit_summary(retrieved_decision)
    else:
        print("[FAILED] Failed to retrieve decision from backend")
        
except Exception as e:
    print(f"[FAILED] Error retrieving decision: {e}")

## Step 8: Regulatory Examiner Simulation

Simulate how a regulatory examiner (like OCC) would query the audit trail.

In [ ]:
print("👨‍**Business:** OCC EXAMINER SIMULATION")
print("=" * 50)

# Simulate a typical examiner query
examiner_query = f"What were the model inputs and adverse action reasons for applicant {applicant_id}?"
print(f"**Analysis:** EXAMINER QUERY: {examiner_query}")

# Generate examiner response from audit trail
examiner_response = backend.format_examiner_response(
    stored_decision_id,
    examiner_query,
    db_backend
)
print(examiner_response)

## Step 9: Decision Replay Validation

Demonstrate that the decision can be validated for consistency and deterministic replay.

In [ ]:
print("🔄 DETERMINISTIC REPLAY VALIDATION")
print("=" * 50)

try:
    # Simulate replay validation by reloading and checking consistency
    replay_decision = db_backend.load_decision(stored_decision_id)

    if replay_decision:
        print("[SUCCESS] Decision replay validation successful")
        print(f"  **Details:** Original decision ID: {stored_decision_id}")
        print(f"  [CONFIG] Decision function: {getattr(replay_decision, 'function_name', 'N/A')}")
        
        # Check if model version is preserved in outputs
        for output in getattr(replay_decision, 'outputs', []):
            if output.name == 'model_version':
                print(f"  **Results:** Model version preserved: {output.value}")
                break
                
        print(f"\n**Insight:** This enables deterministic replay of the exact decision logic")
    else:
        print("[FAILED] Decision replay validation failed")

except Exception as e:
    print(f"[FAILED] Replay error: {e}")

## Step 10: ECOA/Reg B Compliance Validation

Validate that all required regulatory fields are present for compliance.

In [ ]:
print("⚖ REGULATORY COMPLIANCE VALIDATION")
print("=" * 50)

# Define required fields for ECOA/Reg B compliance
required_fields = [
    "regulation",
    "adverse_action_required",
    "bank_owns_model",
    "decision_timestamp"
]

print(f"**Details:** Checking for required ECOA/Reg B fields: {required_fields}")

# Validate regulatory completeness
validation_result = backend.validate_regulatory_completeness(
    retrieved_decision,
    required_fields
)

# Display compliance status
status_icon = "[SUCCESS]" if validation_result['is_compliant'] else "[FAILED]"
status_text = "COMPLIANT" if validation_result['is_compliant'] else "NON-COMPLIANT"

print(f"\n{status_icon} Compliance Status: {status_text}")
print(f"**Results:** Completeness Score: {validation_result['completeness_score']:.1%}")

if validation_result['missing_fields']:
    print(f"[FAILED] Missing Fields: {', '.join(validation_result['missing_fields'])}")
else:
    print(f"[SUCCESS] All required regulatory fields present")

if validation_result['present_fields']:
    print(f"**Details:** Present Fields: {', '.join(validation_result['present_fields'])}")

## Summary

### What We Accomplished
[SUCCESS] **Created a complete ECOA/Reg B compliant audit trail** for an AI credit decision

[SUCCESS] **Captured all required elements:**
- Model inputs and outputs
- Adverse action reason codes
- Decision confidence and model version
- Regulatory metadata

[SUCCESS] **Demonstrated regulatory readiness:**
- Examiner query simulation
- Decision replay capability
- Compliance validation

### Key Benefits
- **Immutable Audit Trail**: Decisions cannot be altered after storage
- **Regulatory Compliance**: Meets ECOA/Reg B requirements automatically
- **Examiner Ready**: Structured responses to regulatory queries
- **Deterministic Replay**: Ability to reconstruct exact decision logic

### Next Steps
In production, you would:
1. Replace the simulated model with your actual ML model
2. Integrate with your loan origination system
3. Set up automated adverse action notice generation
4. Configure regular compliance reporting

**Decision ID for Reference**: `{stored_decision_id}`

## Bitemporal Replay Demonstration

The earlier capture layer records *what* the system did. This section adds
the replay layer — proving *what was known* at decision time, so an auditor
can reconstruct the decision offline against the domain evidence as it
stood on the day of adjudication.

Primitives used here: `BitemporalRecord`, `InMemoryBitemporalStore`,
`append_correction`, `AsOfView`, `PolicyRegistry`, `ExaminerBundle`.

For each primitive in isolation, see [`patterns/`](../../patterns/).
For the same primitives composed into a cross-border payments narrative,
see [`agentic-payments/`](../../agentic-payments/).


### Seed a bitemporal bureau-file store

In [ ]:
# Self-contained imports (safe to re-run after the domain cells above).
import json
from datetime import datetime, timedelta, timezone

from briefcase.bitemporal import (
    AsOfView, BitemporalRecord, InMemoryBitemporalStore, append_correction,
)
from briefcase.compliance import BundleIntegrityError, ExaminerBundle
from briefcase.routing import (
    AgentRoutingDecision, PolicyRegistry, PolicyRule, PolicyVersion,
)

utc = timezone.utc
decision_time = datetime.now(utc) - timedelta(days=90)
correction_time = datetime.now(utc)

# The tradeline that was live at decision time.
bureau = InMemoryBitemporalStore()
tradeline = BitemporalRecord.new(
    key="bureau:tl-7742",
    valid_time=decision_time,
    value={"lender": "OldBank", "balance": 2800, "status": "90_days_late"},
    source="equifax",
    source_trust_level="primary",
    transaction_time=decision_time,
)
bureau.append(tradeline)
print(f"Seeded bureau store: tradeline tl-7742 at {decision_time.date()}")

### FCRA dispute correction

Consumer files an FCRA dispute; the tradeline is removed. The correction is appended with a later `transaction_time`; the original record stays intact.

In [ ]:
append_correction(
    bureau, tradeline,
    corrected_value={"lender": "OldBank", "balance": 0, "status": "removed_per_fcra_dispute"},
    transaction_time=correction_time,
)
print(f"Correction appended at {correction_time.date()}")

### Replay: live vs. as-of decision day

In [ ]:
live = bureau.latest("bureau:tl-7742").value
print(f"Live view (today):    status={live['status']}")
with AsOfView(bureau, transaction_time=decision_time) as view:
    replay = view.latest("bureau:tl-7742").value
    print(f"As-of decision day:   status={replay['status']}")
print("\nAn adverse-action defense reconstructs the bureau file as-of decision date.")

### ExaminerBundle — content-addressed, verifiable, tamper-evident

In [ ]:
policy = PolicyVersion(
    policy_id="credit_underwriting",
    version="1.0.0",
    description="Deny if any tradeline 90+ days delinquent; else grade-based approval.",
    rules=[PolicyRule(
        rule_id="deny_on_delinquency",
        condition={"has_90_day_delinquent": True},
        choice="deny",
        rationale="Derogatory tradeline within look-back window",
    )],
    default_choice="approve",
)
registry = PolicyRegistry()
registry.publish(policy, valid_from=decision_time, transaction_time=decision_time)

routing_decision = AgentRoutingDecision(
    decision_id="replay-demo",
    use_case="credit_underwriting",
    context={"applicant": "app-42"},
    candidates=["approve", "deny"],
    selected="deny",
    policy_id="credit_underwriting",
    policy_version="1.0.0",
    matched_rule_id="deny_on_delinquency",
    evidence_refs=[tradeline.record_id],
    rationale="tradeline tl-7742 90 days delinquent",
    decided_at=decision_time,
)

bundle = ExaminerBundle.build(
    routing_decision,
    evidence_store=bureau,
    policy_registry=registry,
    metadata={"regulation": "ECOA/FCRA"},
)
print(f"content_hash:   {bundle.content_hash}")
print(f"evidence rows:  {len(bundle.evidence)}")

bundle.verify()
print("verify() on untouched bundle:    OK")

payload = bundle.to_json(indent=2)
ExaminerBundle.from_json(payload).verify()
print("verify() after JSON round-trip:  OK")

tampered = json.loads(payload)
tampered["decision"]["selected"] = "approve"
try:
    ExaminerBundle.from_dict(tampered).verify()
except BundleIntegrityError as e:
    print(f"verify() on tampered bundle:     REJECTED ({type(e).__name__})")